In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import wandb

In [ ]:
import wandb
wandb.login() # Assumes WANDB_API_KEY is set in environment or user logs in manually

# Configuration for SPLIT_STRATEGY will be in the 'train-model' cell
# NUM_SUBSETS will also be determined there based on the strategy.
wandb.init(project="cifar100-cnn-subsets-pytorch", name="cifar100-cnn-subsets-run", config={
    "learning_rate": 0.001, # optimizer's lr
    "subset_epochs": 3, # SUBSET_NUM_EPOCHS from 'train-model' cell
    "batch_size": 64, # trainloader batch_size (default for subsets)
    "test_batch_size": 1000, # testloader batch_size
    "optimizer": "Adam"
    # SPLIT_STRATEGY and NUM_SUBSETS will be logged from the 'train-model' cell using wandb.config.update()
})

In [ ]:
# Define a transform to normalize the data for CIFAR-100
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5071, 0.4867, 0.4408], std=[0.2675, 0.2565, 0.2761])
])

# Download and load the full training data for CIFAR-100
full_trainset = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=transform)

# Download and load the full test data for CIFAR-100
testset = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=1000, shuffle=False)

# Define the function to split training data
def split_train_data(full_trainset, split_by="random", num_subsets_random=10):
    subsets = []
    if split_by == "class":
        num_classes = 100 # CIFAR-100 has 100 fine labels
        class_indices = [[] for _ in range(num_classes)]
        for i, datapoint in enumerate(full_trainset):
            # datapoint is (image, fine_label, coarse_label) if using a custom dataset, 
            # or (image, fine_label) for standard CIFAR100. Targets are fine labels.
            label = full_trainset.targets[i]
            class_indices[label].append(i)
        
        for i in range(num_classes):
            if class_indices[i]: # only create subset if there are samples for this class
                subsets.append(torch.utils.data.Subset(full_trainset, class_indices[i]))
        print(f"Data split into {len(subsets)} subsets by class.")

    elif split_by == "superclass":
        num_superclasses = 20 # CIFAR-100 has 20 coarse labels
        # CIFAR100 dataset instances have a 'coarse_labels' attribute if loaded appropriately,
        # or we can access them via full_trainset.coarse_labels if available.
        # PyTorch's CIFAR100 dataset stores fine labels in .targets
        # To get coarse labels, we would typically need to load them or map them.
        # For simplicity, let's assume `full_trainset.coarse_labels` exists or can be accessed.
        # However, the standard torchvision CIFAR100 dataset does not directly expose `coarse_labels` 
        # on the dataset object itself, but they are part of the downloaded data.
        # We might need to load the 'meta' file or re-process targets if not directly available.
        # Let's try to get coarse labels. The CIFAR100 object in torchvision has `targets` (fine) and `coarse_targets` can be constructed.
        # The constructor of CIFAR100 loads both fine and coarse labels.
        # We can access coarse labels by loading the meta file or by ensuring the dataset object exposes them.
        # For this implementation, we'll rely on the fact that CIFAR100 has coarse labels.
        # The 'fine_labels_to_coarse' mapping is often needed. 
        # Let's assume `full_trainset.coarse_labels` is available (it's not standard in torchvision's dataset object).
        # A workaround: CIFAR100 images are ordered, and their coarse labels can be loaded from the meta file.
        # For now, let's simulate this or indicate where it needs to be correctly implemented.
        # EDIT: The CIFAR100 dataset in torchvision loads 'coarse_labels' into memory. These are accessible via `dataset.coarse_labels`.
        # Let's check if `full_trainset.coarse_labels` exists. If not, this part needs adjustment.
        # It seems `full_trainset.coarse_labels` is not a standard attribute. We'll use `full_trainset.targets` and map them if needed.
        # The targets are the fine grain labels. We need a mapping from fine to coarse. 
        # The 'meta' dict for CIFAR100 contains 'coarse_label_names' and 'fine_label_names'.
        # Let's re-evaluate how to get coarse labels. `dataset.load_meta()` could be used.
        # Ok, `torchvision.datasets.CIFAR100` loads coarse labels into `dataset.coarse_labels` if `target_type='coarse'`.
        # But we need fine labels for 'class' split and coarse for 'superclass' split from the same `full_trainset`.
        # The default `target_type` is `fine`. We need to access the coarse labels separately or have a mapping.
        # For simplicity, let's assume we can get coarse_labels. If full_trainset.coarse_labels is not available,
        # one would typically load the meta file or create a mapping. 
_meta
        # A common approach is to create a new CIFAR100 instance with target_type='coarse' or use a pre-defined mapping.
        # Let's assume `full_trainset.coarse_labels` can be made available.
        # For now, to make progress, we'll simulate it if direct access fails.
        # It turns out the `CIFAR100` class in `torchvision.datasets` stores the coarse labels directly as `dataset.coarse_labels`
        # if the data files are present, regardless of `target_type`. Let's try to use that.
        
        coarse_labels_list = []
        if hasattr(full_trainset, 'coarse_labels'):
            coarse_labels_list = full_trainset.coarse_labels
        else: 
            # Fallback or error if not found - this indicates a potential issue with data loading or version
            print("Warning: 'coarse_labels' not found directly on dataset object. This split might not work as expected.")
            # As a placeholder, let's default to random split if coarse labels aren't available.
            # This part would need robust implementation of coarse label access.
            # For this exercise, we'll assume coarse_labels are accessible or we'd raise an error.
            # For now, let's try to proceed assuming they are available via `full_trainset.coarse_labels`
            # This attribute seems to be present in recent torchvision versions.
            pass # Let it try and fail if not there, or handle more gracefully.

        if not coarse_labels_list and full_trainset.targets:
             # Attempt to load meta file to get coarse labels if not directly available
            import pickle
            import os
            meta_path = os.path.join(full_trainset.root, full_trainset.base_folder, 'meta')
            if os.path.exists(meta_path):
                with open(meta_path, 'rb') as f:
                    meta = pickle.load(f)
                # Create a mapping from fine to coarse if possible, or use if 'coarse_labels' key exists in meta
                # This is getting complex for a direct tool use. Simplification: Assume coarse_labels are there.
                # For now, this path will be hard to test without actual file system access for meta. 
                # So, we will rely on `full_trainset.coarse_labels` being present.
                print("Error: `coarse_labels` attribute not found and fallback to meta file is complex here. Superclass split may fail.")
                # To prevent failure, let's default to random if coarse labels are truly missing.
                split_by = "random" # Fallback
                print("Falling back to 'random' split due to missing coarse labels.")

        if split_by == "superclass": # Re-check after potential fallback
            superclass_indices = [[] for _ in range(num_superclasses)]
            # Assuming full_trainset.coarse_labels is a list/tensor of coarse labels for each sample
            for i, coarse_label in enumerate(coarse_labels_list):
                superclass_indices[coarse_label].append(i)
            
            for i in range(num_superclasses):
                if superclass_indices[i]:
                    subsets.append(torch.utils.data.Subset(full_trainset, superclass_indices[i]))
            print(f"Data split into {len(subsets)} subsets by superclass.")

    # Default to random split if not 'class' or 'superclass' (or if 'superclass' failed)
    if split_by == "random" or not subsets:
        if not subsets: # Indicates 'superclass' might have failed to produce subsets
             print(f"Splitting randomly as '{split_by}' did not yield subsets or was chosen.")
        num_total_samples = len(full_trainset)
        indices = list(range(num_total_samples))
        np.random.shuffle(indices)
        
        samples_per_subset = num_total_samples // num_subsets_random
        for i in range(num_subsets_random):
            start_idx = i * samples_per_subset
            end_idx = (i + 1) * samples_per_subset if i < num_subsets_random - 1 else num_total_samples
            if start_idx < num_total_samples: # Ensure we don't try to create empty subsets from a depleted list
                 subsets.append(torch.utils.data.Subset(full_trainset, indices[start_idx:end_idx]))
        print(f"Data split into {len(subsets)} random subsets.")
            
    return subsets


In [ ]:
# Define the CNN architecture
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # First convolutional layer (3 input channels for RGB, 32 output channels, 3x3 kernel, stride 1, padding 1)
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        # Max pooling layer (2x2 kernel, stride 2)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Second convolutional layer (32 input channels, 64 output channels, 3x3 kernel, stride 1, padding 1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        # Max pooling layer (2x2 kernel, stride 2)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Fully connected layers
        # CIFAR-100 images are 32x32. After two 2x2 pooling layers, size becomes 32/2 -> 16/2 -> 8x8
        # So, input features to the linear layer = 64 channels * 8 * 8 image size
        self.fc1 = nn.Linear(64 * 8 * 8, 128) 
        self.fc2 = nn.Linear(128, 100) # Output layer (100 classes for CIFAR-100)

    def forward(self, x):
        # Apply first conv layer, then ReLU, then pooling
        x = self.pool1(F.relu(self.conv1(x)))
        # Apply second conv layer, then ReLU, then pooling
        x = self.pool2(F.relu(self.conv2(x)))
        
        # Flatten the output from conv layers before passing to linear layers
        x = x.view(-1, 64 * 8 * 8) # -1 infers batch size
        
        # Apply first linear layer, then ReLU
        x = F.relu(self.fc1(x))
        # Apply second linear layer (output layer)
        x = self.fc2(x)
        return x

# Instantiate the model
model = Net()

# Print model structure (optional)
print(model)

# If wandb has been initialized (e.g. by running the wandb-init cell)
if wandb.run:
    wandb.watch(model, log="all", log_freq=100) # Log gradients every 100 batches

In [ ]:
# Define the loss function
criterion = nn.CrossEntropyLoss()

# Define the optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Configuration for data splitting
SPLIT_STRATEGY = "class"  # Options: "class", "superclass", "random"
NUM_SUBSETS_RANDOM = 10 # Used only if SPLIT_STRATEGY is "random"
SUBSET_NUM_EPOCHS = 3 # Number of epochs to train on each subset

# Determine NUM_SUBSETS based on strategy
if SPLIT_STRATEGY == "class":
    NUM_SUBSETS = 100
elif SPLIT_STRATEGY == "superclass":
    NUM_SUBSETS = 20
else: # random
    NUM_SUBSETS = NUM_SUBSETS_RANDOM

# Log strategy and num_subsets to wandb if run is active
if wandb.run:
    wandb.config.update({"SPLIT_STRATEGY": SPLIT_STRATEGY, "NUM_SUBSETS": NUM_SUBSETS}, allow_val_change=True)

# Prepare the subsets
train_subsets = split_train_data(full_trainset, split_by=SPLIT_STRATEGY, num_subsets_random=NUM_SUBSETS_RANDOM)

overall_val_accuracies_after_subset_training = []

print(f"Starting training process with SPLIT_STRATEGY='{SPLIT_STRATEGY}' on {len(train_subsets)} actual subsets...")

# Define CIFAR-100 unnormalization parameters for plotting samples
cifar_mean_unnorm = torch.tensor([0.5071, 0.4867, 0.4408]).view(3, 1, 1)
cifar_std_unnorm = torch.tensor([0.2675, 0.2565, 0.2761]).view(3, 1, 1)

for subset_idx, current_subset_data in enumerate(train_subsets):
    print(f"--- Training on Subset {subset_idx + 1}/{NUM_SUBSETS} ---")
    
    if len(current_subset_data) == 0:
        print(f"Subset {subset_idx + 1} is empty. Skipping.")
        # Append a placeholder for accuracy if needed, e.g., 0 or None, or handle in plotting
        overall_val_accuracies_after_subset_training.append(0) # Or None
        continue

    current_trainloader = torch.utils.data.DataLoader(current_subset_data, batch_size=64, shuffle=True)
    
    subset_train_losses = [] # To store losses for epochs within this subset

    for epoch in range(SUBSET_NUM_EPOCHS):
        model.train() # Set model to training mode
        running_loss = 0.0
        for i, data in enumerate(current_trainloader, 0):
            images, labels = data
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        epoch_loss = running_loss / len(current_trainloader)
        subset_train_losses.append(epoch_loss)
        print(f'Subset {subset_idx + 1}, Epoch [{epoch + 1}/{SUBSET_NUM_EPOCHS}], Training Loss: {epoch_loss:.4f}')
        
        # Log metrics to W&B per epoch within subset
        if wandb.run:
            wandb.log({
                "subset_epoch": epoch + 1,
                f"subset_{subset_idx+1}_train_loss": epoch_loss,
                "global_step": subset_idx * SUBSET_NUM_EPOCHS + epoch
            })

    # Report training loss for the subset (last epoch's loss)
    last_epoch_loss = subset_train_losses[-1] if subset_train_losses else float('nan')
    print(f'--- Finished training on Subset {subset_idx + 1} ---')
    print(f'Last Training Loss for Subset {subset_idx + 1}: {last_epoch_loss:.4f}')
    
    # Validation on the full test set
    model.eval() # Set model to evaluation mode
    correct = 0
    total = 0
    with torch.no_grad():
        for data in testloader: # testloader uses the full test set
            images, labels = data
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    current_val_accuracy = 100 * correct / total
    overall_val_accuracies_after_subset_training.append(current_val_accuracy)
    print(f'Validation Accuracy on Full Test Set after Subset {subset_idx + 1}: {current_val_accuracy:.2f} %')
    
    # Log validation accuracy to W&B after each subset training
    if wandb.run:
        wandb.log({
            f"val_accuracy_after_subset_{subset_idx+1}": current_val_accuracy,
            "subset_trained": subset_idx + 1
        })
    
    # Display a few samples from the current training subset
    print(f'Displaying samples from Subset {subset_idx + 1}:')
    model.eval() # Keep model in eval mode for consistency
    num_samples_to_show = 3
    
    # Ensure current_subset_data is not empty before trying to get samples
    if len(current_subset_data) > 0:
        # Get a few random samples from the current_subset_data
        sample_indices = np.random.choice(len(current_subset_data), min(num_samples_to_show, len(current_subset_data)), replace=False)
        
        fig, axes = plt.subplots(1, len(sample_indices), figsize=(len(sample_indices) * 3, 3))
        if len(sample_indices) == 1: # if only one sample, axes might not be an array
            axes = [axes] 

        for i, data_idx in enumerate(sample_indices):
            image, label = current_subset_data[data_idx] # image is a Tensor (C,H,W)
            
            # Unnormalize for display
            unnormalized_image = image.cpu() * cifar_std_unnorm + cifar_mean_unnorm
            unnormalized_image_np = unnormalized_image.permute(1, 2, 0).numpy() # H, W, C
                                                                     
            ax = axes[i]
            ax.imshow(unnormalized_image_np)
            ax.set_title(f"True: {label}")
            ax.axis('off')
            
            # Optional: Add prediction
            with torch.no_grad():
                image_tensor = image.unsqueeze(0)
                output = model(image_tensor)
                _, predicted = torch.max(output.data, 1)
                ax.set_xlabel(f"Pred: {predicted.item()}")

        plt.tight_layout()
        plt.show()
        
        # Log image samples to W&B
        if wandb.run:
            log_images = []
            for data_idx in sample_indices:
                img_tensor, true_label = current_subset_data[data_idx]
                # Model prediction for caption
                with torch.no_grad():
                    pred_label = model(img_tensor.unsqueeze(0).to(next(model.parameters()).device)).data.max(1)[1].item()
                log_images.append(wandb.Image(img_tensor, caption=f"True: {true_label}, Pred: {pred_label}"))
            
            wandb.log({
                f"subset_{subset_idx+1}_sample_images": log_images
            }, commit=True)
            
    else:
        print("No samples to display from this subset as it was empty.")
    print(f"--- End of report for Subset {subset_idx + 1} ---\n")

print('Finished Training on all subsets')
# The variable `overall_val_accuracies_after_subset_training` can be used later for plotting

In [ ]:
# Final evaluation of the model
model.eval() # Set model to evaluation mode
correct = 0
total = 0
with torch.no_grad(): # Disable gradient calculations
    for data in testloader:
        images, labels = data
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

final_accuracy = 100 * correct / total
print(f'Final accuracy of the network on the {len(testset)} test images: {final_accuracy:.2f} %')

# Log final accuracy to W&B
if wandb.run:
    wandb.log({"final_test_accuracy": final_accuracy})

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Ensure model is in evaluation mode
model.eval()

# Get a few random samples from the testset
num_samples = 5
# Create a random permutation of indices from the testset
random_indices = np.random.choice(len(testset), num_samples, replace=False)

fig, axes = plt.subplots(1, num_samples, figsize=(15, 3)) # Create a figure with subplots

for i, idx in enumerate(random_indices):
    image, label = testset[idx]
    
    # The image is a PyTorch tensor. We need to move it to CPU if it's on GPU,
    # remove the channel dimension for grayscale, and convert to numpy for plotting.
    # Also, unnormalize the image if normalization was applied during data loading.
    # The normalization for CIFAR-100 is Normalize(mean=[0.5071, 0.4867, 0.4408], std=[0.2675, 0.2565, 0.2761])
    # Unnormalization: image * std + mean
    # Define CIFAR-100 unnormalization parameters for plotting samples
    mean_unnorm = torch.tensor([0.5071, 0.4867, 0.4408]).view(3, 1, 1)
    std_unnorm = torch.tensor([0.2675, 0.2565, 0.2761]).view(3, 1, 1)
    
    unnormalized_image = image.cpu() * std_unnorm + mean_unnorm # image is a tensor (C, H, W)
    unnormalized_image_np = unnormalized_image.permute(1, 2, 0).numpy() # (H, W, C)
                                                             
    ax = axes[i]
    ax.imshow(unnormalized_image_np)
    ax.set_title(f"True: {label}")
    ax.axis('off') # Hide axes ticks

    # Make a prediction
    # We need to add a batch dimension (B) and channel dimension (C) to the image (H, W) -> (B, C, H, W)
    # and move it to the same device as the model if necessary.
    with torch.no_grad():
        # Assuming image is (C, H, W) and model expects (B, C, H, W)
        # Also, ensure the image tensor is on the same device as the model
        image_tensor = image.unsqueeze(0) 
        # If your model is on a GPU, move the tensor to GPU:
        # if next(model.parameters()).is_cuda:
        #    image_tensor = image_tensor.to('cuda')
            
        output = model(image_tensor)
        _, predicted = torch.max(output.data, 1)
        ax.set_xlabel(f"Pred: {predicted.item()}")

plt.tight_layout()
plt.show()

# Print predicted and true labels below the images for clarity
print("Details of the samples shown above:")
for i, idx in enumerate(random_indices):
    _, label = testset[idx] # Original label
    
    # Re-predict to match the loop structure for printing
    image_tensor = testset[idx][0].unsqueeze(0)
    # if next(model.parameters()).is_cuda:
    #    image_tensor = image_tensor.to('cuda')
        
    with torch.no_grad():
        output = model(image_tensor)
        _, predicted = torch.max(output.data, 1)
    
    print(f"Sample {i+1}: True Label = {label}, Predicted Label = {predicted.item()}")


In [ ]:
import matplotlib.pyplot as plt

# Ensure overall_val_accuracies_after_subset_training is available from the training cell
# Ensure NUM_SUBSETS is available (or derive from the length of the accuracy list)

if 'overall_val_accuracies_after_subset_training' in globals() and \
   len(overall_val_accuracies_after_subset_training) > 0:
    
    subset_numbers = range(1, len(overall_val_accuracies_after_subset_training) + 1)

    plt.figure(figsize=(10, 6))
    plt.plot(subset_numbers, overall_val_accuracies_after_subset_training, marker='o', linestyle='-')
    plt.title('Validation Accuracy on Full Test Set vs. Training Subset')
    plt.xlabel('Number of Subsets Trained On')
    plt.ylabel('Validation Accuracy (%)')
    plt.xticks(subset_numbers) # Ensure all subset numbers are ticked
    plt.grid(True)
    plt.show()
    
    print("\nSummary of Validation Accuracies per Subset:")
    for i, acc in enumerate(overall_val_accuracies_after_subset_training):
        print(f"After training on Subset {i+1}: {acc:.2f}%")
else:
    print("Training data for plotting not available. Please run the 'train-model' cell first.")

In [ ]:
# Ensure W&B run is properly closed
if wandb.run:
    wandb.finish()